# Customer Churn Prediction

**Portfolio category:** Technical demonstration

End-to-end churn classification with class-aware metrics, ROC-AUC and probability-based customer risk ranking.

> This notebook uses simulated/synthetic data where stated and is portfolio evidence, not a claim of client results.


In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score,classification_report,RocCurveDisplay,ConfusionMatrixDisplay
rng=np.random.default_rng(15); n=2500
tenure=rng.integers(1,72,n); monthly=rng.normal(70,22,n).clip(20,180); tickets=rng.poisson(2,n); satisfaction=rng.normal(7,1.5,n).clip(1,10); contract=rng.choice([0,1,2],n,p=[.5,.3,.2])
logit=1.8-.035*tenure+.018*(monthly-70)+.22*tickets-.45*satisfaction-.75*contract
p=1/(1+np.exp(-logit)); churn=rng.binomial(1,p)
df=pd.DataFrame({'tenure':tenure,'monthly_charge':monthly,'support_tickets':tickets,'satisfaction':satisfaction,'contract_type':contract,'churn':churn})
X=df.drop(columns='churn'); y=df.churn; Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
m=RandomForestClassifier(n_estimators=400,max_depth=9,min_samples_leaf=5,class_weight='balanced',random_state=42).fit(Xtr,ytr)
prob=m.predict_proba(Xte)[:,1]; pred=(prob>=.5).astype(int)
print('ROC-AUC:',round(roc_auc_score(yte,prob),3)); print(classification_report(yte,pred,digits=3))
fig,ax=plt.subplots(figsize=(7,5)); RocCurveDisplay.from_predictions(yte,prob,ax=ax); ax.set_title('Customer Churn ROC Curve'); plt.tight_layout()
fig,ax=plt.subplots(figsize=(5,4)); ConfusionMatrixDisplay.from_predictions(yte,pred,ax=ax); ax.set_title('Churn Confusion Matrix'); plt.tight_layout()
importance=pd.Series(m.feature_importances_,index=X.columns).sort_values()
importance.plot(kind='barh',figsize=(7,4),title='Feature Importance'); plt.tight_layout()
risk=df.loc[Xte.index,['tenure','monthly_charge','support_tickets','satisfaction','contract_type']].copy(); risk['churn_probability']=prob; display(risk.sort_values('churn_probability',ascending=False).head(15))


## Evidence produced

This notebook demonstrates reproducible implementation, quantitative evaluation, visualization and interpretation.
